# Registro histórico: 07_vgg16_training

Origem: `C57` — `Colab Notebooks/Vgg-16_Training_1.ipynb`.

VGG16 com duas saídas, label smoothing, sampler e early stopping.

**Limitações conhecidas:** Congelamento indexa parâmetros, não blocos; métricas por lote e relatório global divergem.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
#@title 1) Setup e imports
!nvidia-smi -L || true

import os, random, json, requests
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets
import torchvision.models as models
from torchvision.models import VGG16_Weights

from sklearn.metrics import classification_report, confusion_matrix

# Dispositivo e seeds (reprodutibilidade)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
SEED = 91432
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == "cuda": torch.cuda.manual_seed(SEED)

# (opcional) pequenas otimizações de backend
torch.backends.cudnn.benchmark = True

# Helper functions (igual ao seu script)
if Path("helper_functions.py").is_file():
    print("helper_functions.py já existe, pulando download")
else:
    print("Baixando helper_functions.py …")
    url = "https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py"
    with open("helper_functions.py", "wb") as f:
        f.write(requests.get(url).content)
from helper_functions import accuracy_fn


In [ ]:
#@title 2) Montar Google Drive (se necessário) e definir caminhos
from google.colab import drive

DATA_ROOT = Path("/content/0-vgg")   # igual ao seu
if not DATA_ROOT.exists():
    print("Montando o Google Drive para copiar o dataset …")
    drive.mount('/content/drive')
    !cp -r "/content/drive/MyDrive/0-vgg" "/content/"
else:
    print("Dataset já disponível em /content/0-vgg")

image_path = DATA_ROOT / "base_dir"
train_dir = image_path / "train_dir"
val_dir   = image_path / "val_dir"
test_dir  = image_path / "test_dir"

print(f"Train: {train_dir}\nVal: {val_dir}\nTest: {test_dir}")


In [ ]:
#@title 3) Inspeção do dataset (contagem e amostra)
def count_images(d):
    c = 0
    for root, dirs, files in os.walk(d):
        for f in files:
            if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.gif')):
                c += 1
    return c

print("Imagens:")
print(" - train:", count_images(train_dir))
print(" - val  :", count_images(val_dir))
print(" - test :", count_images(test_dir))

# Amostra visual (opcional)
all_jpgs = list(image_path.glob("*/*/*.jpg"))
if all_jpgs:
    rp = random.choice(all_jpgs)
    img = Image.open(rp)
    plt.figure(figsize=(4,4)); plt.imshow(img); plt.axis("off")
    plt.title(f"Sample: {rp.parent.name}")
    plt.show()

In [ ]:
#@title 4) Data transforms (VGG16 + leve augmentation)
# Mantemos 224x224 + normalização ImageNet para VGG16
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=VGG16_Weights.IMAGENET1K_V1.transforms().mean,
                         std=VGG16_Weights.IMAGENET1K_V1.transforms().std),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=VGG16_Weights.IMAGENET1K_V1.transforms().mean,
                         std=VGG16_Weights.IMAGENET1K_V1.transforms().std),
])

# Datasets
train_data = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_data   = datasets.ImageFolder(root=val_dir,   transform=eval_transform)
test_data  = datasets.ImageFolder(root=test_dir,  transform=eval_transform)

class_names = train_data.classes
class_to_idx = train_data.class_to_idx
print("Classes:", class_names, class_to_idx)


In [ ]:
#@title 5) Dataloaders (com WeightedRandomSampler para o treino)
BATCH_SIZE = 32

# Pesos por classe a partir dos alvos do ImageFolder
targets = torch.tensor(train_data.targets)  # índices das classes
class_sample_count = torch.bincount(targets)
class_weights = 1.0 / class_sample_count.float()
sample_weights = class_weights[targets]

sampler = WeightedRandomSampler(weights=sample_weights.double(),
                                num_samples=len(sample_weights),
                                replacement=True)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2,
                          pin_memory=(device.type=="cuda"),
                          persistent_workers=False)

val_loader = DataLoader(val_data, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2,
                        pin_memory=(device.type=="cuda"))

test_loader = DataLoader(test_data, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2,
                         pin_memory=(device.type=="cuda"))

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")
print("Contagem por classe (train):", class_sample_count.tolist())


In [ ]:
#@title 6) Criar VGG16 para 2 classes (freeze parcial + label smoothing)
def create_vgg16(num_classes=2, pretrained=True, freeze_until_last_block=True):
    if pretrained:
        weights = VGG16_Weights.IMAGENET1K_V1
        model = models.vgg16(weights=weights)
        print("Carregada VGG16 com pesos ImageNet")
    else:
        model = models.vgg16(weights=None)
        print("VGG16 sem pretreino")

    # Congelar features até o último bloco conv (conv5_*)
    if freeze_until_last_block:
        for i, p in enumerate(model.features.parameters()):
            # conv5_* começam em ~idx 24; congelamos <24
            if i < 24:
                p.requires_grad = False

    # Ajustar a última camada para 2 classes
    in_features = model.classifier[6].in_features  # 4096
    model.classifier[6] = nn.Linear(in_features, num_classes)

    # (opcional) compile (PyTorch 2.x + GPU) para speedup
    try:
        model = torch.compile(model)  # ignora se não suportado
        print("torch.compile habilitado")
    except Exception as e:
        pass

    return model

model = create_vgg16(num_classes=2, pretrained=True, freeze_until_last_block=True).to(device)

# Loss (label smoothing ajuda em dados médicos) + AdamW + Scheduler
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                                 patience=3, verbose=True)

print("Parâmetros treináveis:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
#@title 7) Funções de treino, validação e utilitários (AMP + EarlyStopping)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))

def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    pbar = tqdm(loader, leave=False, desc="Treinando")
    for X, y in pbar:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            y_pred = model(X)
            loss = loss_fn(y_pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_acc  += accuracy_fn(y_true=y, y_pred=y_pred.argmax(1))

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss/len(loader), total_acc/len(loader)

@torch.no_grad()
def eval_epoch(model, loader, loss_fn, desc="Validação"):
    model.eval()
    total_loss, total_acc = 0.0, 0.0
    pbar = tqdm(loader, leave=False, desc=desc)
    for X, y in pbar:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
        total_loss += loss.item()
        total_acc  += accuracy_fn(y_true=y, y_pred=y_pred.argmax(1))
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss/len(loader), total_acc/len(loader)

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.05):
        self.best = -np.inf
        self.patience = patience
        self.min_delta = min_delta
        self.bad_epochs = 0
        self.best_state = None

    def step(self, metric, model):
        improved = metric > self.best + self.min_delta
        if improved:
            self.best = metric
            self.bad_epochs = 0
            self.best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            self.bad_epochs += 1
        return improved

    def should_stop(self):
        return self.bad_epochs >= self.patience


In [ ]:
#@title 8) Loop de treino (salva melhor modelo)
EPOCHS   = 50
PATIENCE = 7
MIN_DELTA= 0.05

hist = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}
es = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

start = timer()
for epoch in range(1, EPOCHS+1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, loss_fn)
    vl_loss, vl_acc = eval_epoch(model, val_loader, loss_fn, desc="Validação")

    scheduler.step(vl_loss)

    hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
    hist["val_loss"].append(vl_loss);   hist["val_acc"].append(vl_acc)

    print(f"Train  - Loss: {tr_loss:.4f} | Acc: {tr_acc:.2f}%")
    print(f"Val    - Loss: {vl_loss:.4f} | Acc: {vl_acc:.2f}%")
    print(f"LR atual: {optimizer.param_groups[0]['lr']:.6f}")

    improved = es.step(vl_acc, model)
    if improved:
        print(f"🎯 Novo melhor Val Acc: {vl_acc:.2f}%")
    else:
        print(f"⏳ Sem melhora por {es.bad_epochs}/{PATIENCE} épocas")

    if es.should_stop():
        print("🛑 Early stopping acionado")
        break

total_time = timer() - start
print(f"\nTempo total de treino: {total_time:.2f}s")

# Carregar melhor estado
if es.best_state is not None:
    model.load_state_dict(es.best_state)
    best_val_acc = es.best
else:
    best_val_acc = hist["val_acc"][-1]


In [ ]:
#@title 9) Plot histórico (loss e acc)
epochs_range = range(1, len(hist["train_loss"])+1)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1); plt.plot(epochs_range, hist["train_loss"], label="train"); plt.plot(epochs_range, hist["val_loss"], label="val")
plt.title("Loss"); plt.xlabel("Época"); plt.legend(); plt.grid(alpha=0.3)
plt.subplot(1,2,2); plt.plot(epochs_range, hist["train_acc"], label="train"); plt.plot(epochs_range, hist["val_acc"], label="val")
plt.title("Accuracy (%)"); plt.xlabel("Época"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
#@title 10) Avaliação no TESTE + relatório e matriz de confusão
@torch.no_grad()
def evaluate_full(model, loader):
    model.eval()
    y_true, y_pred = [], []
    total_loss, total_acc = 0.0, 0.0
    pbar = tqdm(loader, desc="Testando", leave=False)
    for X, y in pbar:
        X, y = X.to(device), y.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(X)
            loss   = loss_fn(logits, y)
        total_loss += loss.item()
        preds = logits.argmax(1)
        total_acc  += accuracy_fn(y_true=y, y_pred=preds)
        y_true.extend(y.cpu().tolist()); y_pred.extend(preds.cpu().tolist())
    return (total_loss/len(loader), total_acc/len(loader), np.array(y_true), np.array(y_pred))

test_loss, test_acc, y_true, y_pred = evaluate_full(model, test_loader)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

print("\n📈 Classification report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4,4))
plt.imshow(cm, cmap="Blues"); plt.title("Confusion Matrix"); plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=45); plt.yticks(range(len(class_names)), class_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha="center", va="center")
plt.xlabel("Pred"); plt.ylabel("True"); plt.tight_layout(); plt.show()


In [ ]:
#@title 11) Salvar modelo e histórico (local e Google Drive)
LOCAL_DIR = Path("/content/0-vgg/models"); LOCAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR = Path("/content/drive/MyDrive/0-vgg/models")  # se preferir salvar no Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    save_to_drive = True
except Exception:
    save_to_drive = False

MODEL_NAME = "VGG16_SkinLesion_Binary_Classifier_1.pth"
history_name = "training_history_vgg16_1.json"

# Evitar overwrite: enumerar
def next_path(base_dir, name):
    p = base_dir / name
    if not p.exists(): return p
    stem, ext = p.stem, p.suffix
    k = 1
    while True:
        pp = base_dir / f"{stem}_{k}{ext}"
        if not pp.exists(): return pp
        k += 1

model_local = next_path(LOCAL_DIR, MODEL_NAME)
torch.save(model.state_dict(), model_local)
print("Modelo salvo em:", model_local)

if save_to_drive:
    model_drive = next_path(DRIVE_DIR, MODEL_NAME)
    torch.save(model.state_dict(), model_drive)
    print("Modelo salvo no Drive:", model_drive)

history = {
    "train_losses": hist["train_loss"],
    "val_losses":   hist["val_loss"],
    "train_accs":   hist["train_acc"],
    "val_accs":     hist["val_acc"],
    "best_val_acc": float(best_val_acc),
    "test_loss":    float(test_loss),
    "test_acc":     float(test_acc),
    "epochs_run":   len(hist["train_loss"]),
    "total_train_time_sec": total_time,
    "classes": class_names,
}
hist_local = next_path(LOCAL_DIR, history_name)
with open(hist_local, "w") as f: json.dump(history, f, indent=2)
print("Histórico salvo em:", hist_local)

if save_to_drive:
    hist_drive = next_path(DRIVE_DIR, history_name)
    with open(hist_drive, "w") as f: json.dump(history, f, indent=2)
    print("Histórico salvo no Drive:", hist_drive)


In [ ]:
#@title 13) Sumário final
print("="*60)
print("📊 TRAINING SUMMARY")
print("="*60)
print(f"Model: VGG16")
print(f"Task: Binary Classification (Melanoma vs Non-Melanoma)")
print(f"Dataset Size: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")
print(f"Training Time: {total_time:.2f} seconds")
print(f"Epochs Trained: {len(hist['train_loss'])}")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print(f"Model Saved (local): {model_local}")
print("="*60)


In [ ]:
print(model.state_dict())